In [0]:
catalog = "workspace"
schema = "retail_lab"
base = f"/Volumes/{catalog}/{schema}/raw_files"




In [0]:
customers = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base}/customers.csv"))

products = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base}/products.csv"))

orders = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base}/orders.csv"))

display(customers.limit(5))
display(products.limit(5))
display(orders.limit(5))

customer_id,customer_name,email,city,region,signup_date,customer_segment
C0001,Customer_001,customer1@example.com,Guwahati,East,2023-03-31,Small Business
C0002,Customer_002,customer2@example.com,Pune,West,2025-02-12,Small Business
C0003,Customer_003,customer3@example.com,Kochi,South,2024-10-16,Small Business
C0004,Customer_004,customer4@example.com,Surat,West,2024-03-14,Corporate
C0005,Customer_005,customer5@example.com,Nagpur,Central,2024-03-09,Corporate


product_id,product_name,category,unit_cost,list_price
P001,Wireless Mouse,Electronics,8047.36,8559.07
P002,Keyboard,Electronics,6817.61,13805.23
P003,Monitor,Electronics,7645.84,10031.39
P004,Headphones,Electronics,6745.75,5575.51
P005,Webcam,Electronics,447.35,7945.41


order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,order_status,shipping_date,payment_method
O00001,2025-04-18,C0032,P017,7,6951.23,0.0,Cancelled,null,UPI
O00002,2026-03-17,C0089,P006,1,7442.24,0.0,Delivered,2026-03-21,Cash on Delivery
O00003,2026-07-23,C0087,P007,6,4325.82,0.1,Pending,null,UPI
O00004,2024-01-21,C0092,P011,6,7404.01,0.1,Delivered,2024-01-25,UPI
O00005,2024-04-08,C0078,P014,1,8394.51,0.15,Pending,null,Cash on Delivery


In [0]:
parquet_base = f"/Volumes/{catalog}/{schema}/raw_files/parquet_output"

customers.write.mode("overwrite").parquet(f"{parquet_base}/customers")
products.write.mode("overwrite").parquet(f"{parquet_base}/products")
orders.write.mode("overwrite").parquet(f"{parquet_base}/orders")

print("Successfully converted all datasets to Parquet inside your existing volume!")

Successfully converted all datasets to Parquet inside your existing volume!


In [0]:
orders_parquet = spark.read.parquet(f"{parquet_base}/orders")

orders_parquet.printSchema()
display(orders_parquet.limit(10))

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- order_status: string (nullable = true)
 |-- shipping_date: date (nullable = true)
 |-- payment_method: string (nullable = true)



order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,order_status,shipping_date,payment_method
O00001,2025-04-18,C0032,P017,7,6951.23,0.0,Cancelled,null,UPI
O00002,2026-03-17,C0089,P006,1,7442.24,0.0,Delivered,2026-03-21,Cash on Delivery
O00003,2026-07-23,C0087,P007,6,4325.82,0.1,Pending,null,UPI
O00004,2024-01-21,C0092,P011,6,7404.01,0.1,Delivered,2024-01-25,UPI
O00005,2024-04-08,C0078,P014,1,8394.51,0.15,Pending,null,Cash on Delivery
O00006,2025-03-06,C0023,P007,5,4325.82,0.0,Delivered,2025-03-11,UPI
O00007,2024-04-17,C0036,P020,7,17319.87,0.1,Delivered,2024-04-21,Net Banking
O00008,2024-12-25,C0019,P021,3,12639.77,0.05,Shipped,2024-12-29,Debit Card
O00009,2026-03-21,C0081,P011,4,7404.01,0.2,Delivered,2026-03-23,Net Banking
O00010,2025-08-14,C0115,P007,4,4325.82,0.1,Delivered,null,Net Banking


In [0]:
customers_parquet = spark.read.parquet(f"{parquet_base}/customers")
products_parquet = spark.read.parquet(f"{parquet_base}/products")
orders_parquet = spark.read.parquet(f"{parquet_base}/orders")

customers_parquet.createOrReplaceTempView("customers")
products_parquet.createOrReplaceTempView("products")
orders_parquet.createOrReplaceTempView("orders")

print("Temporary views 'customers', 'products', and 'orders' have been successfully registered!")

Temporary views 'customers', 'products', and 'orders' have been successfully registered!


In [0]:
# Set as dataframes:
df_orders = spark.read.parquet(f"{parquet_base}/orders")
df_products = spark.read.parquet(f"{parquet_base}/products")
df_customers = spark.read.parquet(f"{parquet_base}/customers")

# Display first 10 orders:
print("First 10 orders: ")
df_orders_10 = df_orders.limit(10).show()
print(df_orders_10)

# Show the schema and row count of each dataset:
print("Schema: ")
df_orders.printSchema()
df_products.printSchema()
df_customers.printSchema()

# Find all orders with status Delivered
from pyspark.sql.functions import col, upper
print("Delivered orders: ")
df_del_orders = df_orders.filter(upper(col("order_status")) == "DELIVERED")
df_del_orders.show()

# Show orders placed in 2025
print("Orders in 2025: ")
df_orders_2025 = df_orders.filter(col("order_date").like("%2025%"))
df_orders_2025.show()

# List all distinct order statuses
print("Distinct order statuses: ")
df_distinct_status = df_orders.groupBy("order_status").count().orderBy("count", ascending=False)
df_distinct_status.show()

# ORders with missing Shipping date:
print("Orders with missing date: ")
df_missing_date = df_orders.filter(col("shipping_date").isNull())
df_missing_date.show()
print(f"Total number of orders with missing date:{df_missing_date.select('order_id').count()}")


First 10 orders: 
+--------+----------+-----------+----------+--------+----------+------------+------------+-------------+----------------+
|order_id|order_date|customer_id|product_id|quantity|unit_price|discount_pct|order_status|shipping_date|  payment_method|
+--------+----------+-----------+----------+--------+----------+------------+------------+-------------+----------------+
|  O00001|2025-04-18|      C0032|      P017|       7|   6951.23|         0.0|   Cancelled|         NULL|             UPI|
|  O00002|2026-03-17|      C0089|      P006|       1|   7442.24|         0.0|   Delivered|   2026-03-21|Cash on Delivery|
|  O00003|2026-07-23|      C0087|      P007|       6|   4325.82|         0.1|     Pending|         NULL|             UPI|
|  O00004|2024-01-21|      C0092|      P011|       6|   7404.01|         0.1|   Delivered|   2024-01-25|             UPI|
|  O00005|2024-04-08|      C0078|      P014|       1|   8394.51|        0.15|     Pending|         NULL|Cash on Delivery|
|  O00

In [0]:
#imports
from pyspark.sql.functions import col, year, month, sum, coalesce, lit, round

# Total number of orders by status
print("Total number of orders by status: ")
df_distinct_status2 = df_orders.groupBy("order_status").count().orderBy("count", ascending=False)
df_distinct_status2.show()

#Total quantity sold by product
print("Total quantity sold by product: ")
df_quantity_sold = df_orders.groupBy("product_id").sum("quantity").orderBy("sum(quantity)", ascending=False)
df_quantity_sold.show()

# Revenue by payment method. Revenue = quantity × unit_price × (1 − discount_pct). Treat a missing discount as 0
df_revenue_per_method = df_orders.withColumn("revenue", col("quantity") * col("unit_price") * (1 - col("discount_pct")))
df_revenue_per_method = df_revenue_per_method.groupBy("payment_method").sum("revenue").orderBy("sum(revenue)", ascending=False)
df_revenue_per_method.show()

# Find monthly revenue for 2025
df_with_revenue = df_orders.withColumn("calculated_revenue",col("quantity").cast("double") * col("unit_price").cast("double") * (1 - coalesce(col("discount_pct").cast("double"), lit(0.0)))) #cast("double") for numeric

df_revenue_per_month = (df_with_revenue.filter(year(col("order_date")) == 2025).groupBy(month(col("order_date")).alias("month")).agg(sum("calculated_revenue").alias("total_monthly_revenue"))
.orderBy("month"))

df_revenue_per_month.show()

# Top 5 customers by total order value
df_top5 = df_with_revenue.groupBy("customer_id").agg(round(sum("calculated_revenue"), 2).alias("total_order_value")).orderBy("total_order_value", ascending=False).limit(5)
df_top5.show()

Total number of orders by status: 
+------------+-----+
|order_status|count|
+------------+-----+
|   Delivered|  938|
|     Shipped|  228|
|     Pending|  138|
|   Cancelled|  114|
|    Returned|   82|
+------------+-----+

Total quantity sold by product: 
+----------+-------------+
|product_id|sum(quantity)|
+----------+-------------+
|      P003|          359|
|      P011|          358|
|      P021|          348|
|      P015|          331|
|      P010|          299|
|      P001|          299|
|      P012|          292|
|      P006|          290|
|      P014|          289|
|      P008|          287|
|      P007|          281|
|      P002|          275|
|      P005|          271|
|      P016|          266|
|      P018|          261|
|      P013|          255|
|      P004|          239|
|      P017|          238|
|      P009|          231|
|      P019|          226|
+----------+-------------+
only showing top 20 rows
+----------------+--------------------+
|  payment_method|        sum

In [0]:
%sql
-- Join orders to customers and show order ID, customer name, city and region
select o.order_id, c.customer_name, c.city, c.region from orders as o join customers as c on o.customer_id = c.customer_id;


order_id,customer_name,city,region
O00001,Customer_032,Delhi,North
O00002,Customer_089,Guwahati,East
O00003,Customer_087,Hyderabad,South
O00004,Customer_092,Delhi,North
O00005,Customer_078,Raipur,Central
O00006,Customer_023,Ahmedabad,West
O00007,Customer_036,Raipur,Central
O00008,Customer_019,Chennai,South
O00009,Customer_081,Mumbai,West
O00010,Customer_115,Surat,West


In [0]:
%sql
-- Join orders to products and calculate revenue by product category
select p.category, sum((o.quantity * o.unit_price)*(1-o.discount_pct)) as revenue from orders as o right join products as p on o.product_id = p.product_id group by p.category;

category,revenue
Stationery,9249234.277499992
Electronics,1.4000292753999999E7
Furniture,1.1072197099499991E7
Accessories,1.1787900921499988E7


In [0]:
%sql
-- Find customers who have never placed an order.
select c.customer_id, coalesce(o.order_id, 'No orders') as NoOrderConfirmation from customers as c left join orders as o on c.customer_id = o.customer_id where o.order_id is null;

customer_id,NoOrderConfirmation


In [0]:
%sql
-- duplicate order IDs
select order_id, count(order_id) as rep_times from orders group by order_id having count(order_id) > 1 order by rep_times asc;

order_id,rep_times
